In [25]:
# Cell 1: Install packages and Cloudflare Tunnel
!pip install -q streamlit plotly pyngrok
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Environment & Tunneling Ready!")

✅ Environment & Tunneling Ready!


In [26]:
# Cell 2: Auto-detect files from your previous runs
from pathlib import Path

splat_ply = next(Path("/kaggle").rglob("ceramic_splatfacto*.ply"), None)
nerf_ply = next(Path("/kaggle").rglob("point_cloud.ply"), None)
results_csv = next(Path("/kaggle").rglob("results.csv"), None)

print("🔍 Detected Files in Kaggle:")
print(" - Splatfacto 3D Model (.ply) :", splat_ply or "Not found (Demo mode will be used)")
print(" - Nerfacto Point Cloud (.ply):", nerf_ply or "Not found (Demo mode will be used)")
print(" - Metrics results.csv        :", results_csv or "Built-in verified metrics will be used")

🔍 Detected Files in Kaggle:
 - Splatfacto 3D Model (.ply) : /kaggle/input/datasets/dhyani1611/ceramic-araatifact-3d-models/ceramic_splatfacto.ply
 - Nerfacto Point Cloud (.ply): /kaggle/input/datasets/dhyani1611/ceramic-araatifact-3d-models/point_cloud.ply
 - Metrics results.csv        : Built-in verified metrics will be used


In [27]:
%%writefile app.py
import os
import json
import base64
from pathlib import Path
import numpy as np
import pandas as pd
import streamlit as st
import streamlit.components.v1 as components
import plotly.graph_objects as go

st.set_page_config(
    page_title="3D Cultural Artifact Preservation | Final Model",
    page_icon="🏺",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Dark preservation theme styling
st.markdown("""
<style>
    .stApp { background-color: #0b1320; color: #e6edf3; }
    .metric-card {
        background: linear-gradient(135deg, #132238 0%, #1a2f4c 100%);
        border: 1px solid #233e66; border-radius: 10px; padding: 14px 18px; margin-bottom: 10px;
    }
    .metric-title { color: #8da4c4; font-size: 12px; font-weight: 600; text-transform: uppercase; }
    .metric-value { color: #ffffff; font-size: 24px; font-weight: 700; margin: 3px 0; }
    .metric-delta { color: #3fb950; font-size: 12px; font-weight: 600; }
</style>
""", unsafe_allow_html=True)

# 1. Auto-locate default models (Nerfacto first, then Splatfacto)
nerf_auto = None
for name in ["point_cloud_vase_500k.ply", "point_cloud*.ply"]:
    matches = list(Path("/kaggle").rglob(name)) if Path("/kaggle").exists() else []
    matches += list(Path(".").rglob(name))
    if matches:
        nerf_auto = matches[0]
        break

splat_auto = None
for name in ["ceramic_splatfacto_vase.ply", "ceramic_splatfacto_v2.ply", "ceramic_splatfacto*.ply"]:
    matches = list(Path("/kaggle").rglob(name)) if Path("/kaggle").exists() else []
    matches += list(Path(".").rglob(name))
    if matches:
        splat_auto = matches[0]
        break

# 2. Sidebar with Nerfacto first, then Splatfacto
with st.sidebar:
    st.image("https://img.icons8.com/fluency/96/vase.png", width=54)
    st.title("Preservation Hub")
    st.caption("Default Models: point_cloud_vase_500k.ply & ceramic_splatfacto_vase.ply")
    st.markdown("---")
    
    st.subheader("📁 Upload Your 3D Files")
    multi_uploaded = st.file_uploader(
        "Drag & Drop BOTH .ply files together:",
        type=["ply"],
        accept_multiple_files=True,
        help="Upload point_cloud_vase_500k.ply and ceramic_splatfacto_vase.ply simultaneously"
    )
    with st.expander("Or upload individually:"):
        upload_nerf = st.file_uploader("1. Nerfacto (Points .ply):", type=["ply"], key="u_nerf")
        upload_splat = st.file_uploader("2. Splatfacto v2 (3DGS .ply):", type=["ply"], key="u_splat")

# 3. Resolve model bytes (Nerfacto first, then Splatfacto)
nerf_bytes = None
splat_bytes = None

if multi_uploaded:
    for f in multi_uploaded:
        fn = f.name.lower()
        if "nerf" in fn or "point" in fn:
            nerf_bytes = f.getvalue()
            st.sidebar.success(f"Loaded NeRF: {f.name}")
        elif "splat" in fn or "vase" in fn:
            splat_bytes = f.getvalue()
            st.sidebar.success(f"Loaded 3DGS: {f.name}")
        else:
            if nerf_bytes is None: nerf_bytes = f.getvalue()
            elif splat_bytes is None: splat_bytes = f.getvalue()

if upload_nerf: nerf_bytes = upload_nerf.getvalue()
if upload_splat: splat_bytes = upload_splat.getvalue()

# Fallback to default auto-discovered files
if nerf_bytes is None and nerf_auto:
    nerf_bytes = nerf_auto.read_bytes()
if splat_bytes is None and splat_auto:
    splat_bytes = splat_auto.read_bytes()

# Encode to base64 for WebGL renderer
nerf_b64 = base64.b64encode(nerf_bytes).decode("ascii") if nerf_bytes else ""
splat_b64 = base64.b64encode(splat_bytes).decode("ascii") if splat_bytes else ""

# Status in sidebar (Nerfacto first, then Splatfacto)
st.sidebar.markdown("---")
st.sidebar.markdown(f"**1. Nerfacto Points:** {'`Loaded ✅`' if nerf_bytes else '`Demo Mode`'}")
if nerf_auto and not multi_uploaded and not upload_nerf:
    st.sidebar.caption(f"Default: {nerf_auto.name} ({len(nerf_bytes)/1024**2:.2f} MB)")

st.sidebar.markdown(f"**2. Splatfacto v2:** {'`Loaded ✅`' if splat_bytes else '`Demo Mode`'}")
if splat_auto and not multi_uploaded and not upload_splat:
    st.sidebar.caption(f"Default: {splat_auto.name} ({len(splat_bytes)/1024**2:.2f} MB)")

# Main Header (NeRF first, then Splatfacto)
st.title("🏺 Digital Preservation of Cultural Artifacts")
st.markdown("Final Retrained 3D Models on `binh_gom_1`: **Nerfacto (1M Points)** vs **Splatfacto v2 (15k iters + OBJ Prior)**")

# Top KPI Metric Cards
c1, c2, c3, c4 = st.columns(4)
with c1:
    st.markdown('<div class="metric-card"><div class="metric-title">Final Peak PSNR</div><div class="metric-value">24.68 dB</div><div class="metric-delta">▲ +12.70 dB vs NeRF (+106.1%)</div></div>', unsafe_allow_html=True)
with c2:
    st.markdown('<div class="metric-card"><div class="metric-title">Final Structural SSIM</div><div class="metric-value">0.9349</div><div class="metric-delta">▲ +0.2807 vs NeRF (+42.9%)</div></div>', unsafe_allow_html=True)
with c3:
    st.markdown('<div class="metric-card"><div class="metric-title">Final Perceptual LPIPS</div><div class="metric-value">0.1067</div><div class="metric-delta">▼ -80.1% error reduction</div></div>', unsafe_allow_html=True)
with c4:
    st.markdown('<div class="metric-card"><div class="metric-title">Final Real-Time FPS</div><div class="metric-value">22.88 FPS</div><div class="metric-delta">⚡ 217.9x faster rendering</div></div>', unsafe_allow_html=True)

# Navigation Tabs
tab_viewer, tab_bench, tab_retrain, tab_renders = st.tabs([
    "🌐 3D Web Viewer (Both Models)",
    "📊 Final Benchmark Comparison",
    "🔬 Retraining Workflow (v1 vs v2)",
    "🖼️ Held-Out Views Gallery"
])

# TAB 1: 3D DUAL VIEWER (Nerfacto first, then Splatfacto)
with tab_viewer:
    st.subheader("Dual Model 3D WebGL Viewer (Three.js + Spark.js)")
    st.caption("Switch between Nerfacto (point_cloud_vase_500k.ply), Splatfacto v2 (ceramic_splatfacto_vase.ply), and Procedural Demo.")
    viewer_html = f"""
    <!DOCTYPE html>
    <html>
    <head>
    <meta charset="UTF-8">
    <style>
      body {{ margin:0; background:#080e18; color:white; font-family:sans-serif; overflow:hidden; }}
      .bar {{ display:flex; gap:8px; padding:10px 14px; background:#101c2d; border-bottom:1px solid #1f3352; align-items:center; }}
      button {{ padding:7px 14px; background:#18283f; color:#e2ecf8; border:1px solid #2b456b; border-radius:6px; cursor:pointer; font-size:12px; font-weight:500; transition:0.2s; }}
      button:hover {{ background:#253d5e; }}
      button.active {{ background:#1f6feb; border-color:#58a6ff; color:white; box-shadow:0 0 10px rgba(31,111,235,0.4); }}
      #viewer {{ width:100%; height:540px; position:relative; }}
      .hud {{ position:absolute; top:12px; right:14px; background:rgba(13,22,36,0.85); backdrop-filter:blur(6px); padding:10px 14px; border-radius:8px; font-size:11px; color:#79c0ff; border:1px solid #388bfd40; pointer-events:none; }}
    </style>
    <script type="importmap">
    {{
      "imports": {{
        "three": "https://cdn.jsdelivr.net/npm/three@0.180.0/build/three.module.js",
        "three/addons/": "https://cdn.jsdelivr.net/npm/three@0.180.0/examples/jsm/",
        "@sparkjsdev/spark": "https://sparkjs.dev/releases/spark/2.4.0/spark.module.js"
      }}
    }}
    </script>
    </head>
    <body>
    <div class="bar">
      <button id="nerfBtn" class="active">🔵 Nerfacto (Point Cloud)</button>
      <button id="splatBtn">✨ Splatfacto v2 (3DGS)</button>
      <button id="demoBtn">🏺 Procedural Demo</button>
      <div style="flex:1"></div>
      <button id="frontBtn">Front</button><button id="topBtn">Top</button><button id="sideBtn">Side</button><button id="resetBtn">Reset</button>
      <button id="rotBtn">🔄 Orbit: Off</button>
    </div>
    <div id="viewer"><div class="hud" id="hud">Loading Models...</div></div>

    <script type="module">
      import * as THREE from "three";
      import {{ OrbitControls }} from "three/addons/controls/OrbitControls.js";
      import {{ PLYLoader }} from "three/addons/loaders/PLYLoader.js";
      import {{ SparkRenderer, SplatMesh }} from "@sparkjsdev/spark";

      const host = document.getElementById("viewer");
      const hud = document.getElementById("hud");
      const scene = new THREE.Scene();
      scene.background = new THREE.Color(0x080e18);
      const camera = new THREE.PerspectiveCamera(48, host.clientWidth / host.clientHeight, 0.001, 1000);
      const renderer = new THREE.WebGLRenderer({{ antialias:true }});
      renderer.setSize(host.clientWidth, host.clientHeight);
      host.appendChild(renderer.domElement);

      let spark = null;
      try {{ spark = new SparkRenderer({{ renderer }}); scene.add(spark); }} catch(e) {{}}
      const controls = new OrbitControls(camera, renderer.domElement);
      controls.enableDamping = true;
      scene.add(new THREE.GridHelper(3, 30, 0x1f6feb, 0x15273f));
      scene.add(new THREE.AmbientLight(0xffffff, 1.2));

      let nerfObj = null, splatObj = null, demoObj = null, autoRot = false;
      const nerfData = "{nerf_b64}";
      const splatData = "{splat_b64}";

      function createDemo() {{
        const n = 12000, geo = new THREE.BufferGeometry(), pos = new Float32Array(n*3), col = new Float32Array(n*3);
        for(let i=0; i<n; i++) {{
          const t = Math.random(), th = Math.random() * Math.PI * 2;
          const r = 0.22 + 0.35 * Math.sin(Math.PI * Math.pow(t, 0.8)) - 0.12 * Math.cos(3*Math.PI*t);
          pos[i*3] = r * Math.cos(th); pos[i*3+1] = (t-0.5)*1.4; pos[i*3+2] = r * Math.sin(th);
          col[i*3] = 0.2 + 0.5*(1-t); col[i*3+1] = 0.4 + 0.5*(1-t); col[i*3+2] = 0.7 + 0.3*t;
        }}
        geo.setAttribute("position", new THREE.BufferAttribute(pos, 3));
        geo.setAttribute("color", new THREE.BufferAttribute(col, 3));
        demoObj = new THREE.Points(geo, new THREE.PointsMaterial({{ size:0.012, vertexColors:true }}));
        scene.add(demoObj);
      }}

      async function init() {{
        createDemo();
        // 1. Load Nerfacto point cloud FIRST as default active
        if (nerfData) {{
          try {{
            const geo = await new PLYLoader().loadAsync("data:application/octet-stream;base64," + nerfData);
            nerfObj = new THREE.Points(geo, new THREE.PointsMaterial({{ size:0.005, vertexColors:true }}));
            nerfObj.visible = true;
            demoObj.visible = false;
            scene.add(nerfObj);
            hud.textContent = "Mode: Nerfacto (point_cloud_vase_500k.ply)";
          }} catch(e) {{ console.log(e); }}
        }}
        // 2. Load Splatfacto v2
        if (splatData) {{
          try {{
            splatObj = new SplatMesh({{ url: "data:application/octet-stream;base64," + splatData, lod: true }});
            splatObj.visible = !nerfData;
            scene.add(splatObj);
            await splatObj.initialized;
            if (!nerfData) {{
              demoObj.visible = false;
              hud.textContent = "Mode: Splatfacto v2 (ceramic_splatfacto_vase.ply)";
              setMode("splat");
            }}
          }} catch(e) {{ console.log(e); }}
        }}
        camera.position.set(0, 0.2, 2.5);
      }}

      function setMode(m) {{
        document.getElementById("nerfBtn").classList.toggle("active", m==="nerf");
        document.getElementById("splatBtn").classList.toggle("active", m==="splat");
        document.getElementById("demoBtn").classList.toggle("active", m==="demo");
        if(nerfObj) nerfObj.visible = (m==="nerf");
        if(splatObj) splatObj.visible = (m==="splat");
        if(demoObj) demoObj.visible = (m==="demo" || (!nerfObj && m==="nerf") || (!splatObj && m==="splat"));
        hud.textContent = `Mode: ${{m === "nerf" ? "Nerfacto (Point Cloud)" : m === "splat" ? "Splatfacto v2 (3DGS)" : "Procedural Demo"}}`;
      }}

      document.getElementById("nerfBtn").onclick = () => setMode("nerf");
      document.getElementById("splatBtn").onclick = () => setMode("splat");
      document.getElementById("demoBtn").onclick = () => setMode("demo");
      document.getElementById("frontBtn").onclick = () => {{ camera.position.set(0, 0.2, 2.5); controls.target.set(0,0,0); }};
      document.getElementById("topBtn").onclick = () => {{ camera.position.set(0.001, 2.8, 0); controls.target.set(0,0,0); }};
      document.getElementById("sideBtn").onclick = () => {{ camera.position.set(2.8, 0.1, 0); controls.target.set(0,0,0); }};
      document.getElementById("resetBtn").onclick = () => {{ camera.position.set(0, 0.2, 2.5); controls.reset(); }};
      document.getElementById("rotBtn").onclick = (e) => {{
        autoRot = !autoRot; controls.autoRotate = autoRot;
        e.currentTarget.textContent = autoRot ? "🔄 Orbit: On" : "🔄 Orbit: Off";
        e.currentTarget.classList.toggle("active", autoRot);
      }};

      window.onresize = () => {{
        camera.aspect = host.clientWidth / host.clientHeight;
        camera.updateProjectionMatrix();
        renderer.setSize(host.clientWidth, host.clientHeight);
      }};

      let frames = 0, last = performance.now();
      renderer.setAnimationLoop(() => {{
        controls.update();
        renderer.render(scene, camera);
        frames++;
        const now = performance.now();
        if(now - last >= 1000) {{
          const fps = Math.round((frames * 1000)/(now - last));
          hud.textContent = hud.textContent.split("|")[0] + `| FPS: ${{fps}}`;
          frames = 0; last = now;
        }}
      }});
      init();
    </script>
    </body>
    </html>
    """
    components.html(viewer_html, height=580, scrolling=False)
    st.caption("Controls: Left-drag to Orbit · Right-drag to Pan · Scroll wheel to Zoom.")

# TAB 2: FINAL BENCHMARK COMPARISON
with tab_bench:
    st.subheader("Quantitative Evaluation Results (18 Held-Out Views)")
    st.caption("Comparing Baseline NeRF vs Splatfacto v1 vs Final Retrained Splatfacto v2 (15,000 steps + OBJ prior).")
    
    df = pd.DataFrame([
        {"Metric": "PSNR (dB)", "Nerfacto (NeRF)": 11.9713, "Splatfacto v1 (5k)": 20.1740, "Splatfacto v2 (FINAL 15k)": 24.6751, "Final Gain": "+12.70 dB (+106.1%) 🏆"},
        {"Metric": "SSIM", "Nerfacto (NeRF)": 0.6542, "Splatfacto v1 (5k)": 0.8172, "Splatfacto v2 (FINAL 15k)": 0.9349, "Final Gain": "+0.2807 (+42.9%) 🏆"},
        {"Metric": "LPIPS", "Nerfacto (NeRF)": 0.5364, "Splatfacto v1 (5k)": 0.2979, "Splatfacto v2 (FINAL 15k)": 0.1067, "Final Gain": "-80.1% error reduction 🏆"},
        {"Metric": "FPS (Evaluation)", "Nerfacto (NeRF)": 0.1050, "Splatfacto v1 (5k)": 17.3500, "Splatfacto v2 (FINAL 15k)": 22.8814, "Final Gain": "217.9x faster rendering ⚡"},
        {"Metric": "Ray Throughput", "Nerfacto (NeRF)": "6,881 /s", "Splatfacto v1 (5k)": "1.14M /s", "Splatfacto v2 (FINAL 15k)": "1.50M /s", "Final Gain": "1.50 Million rays/sec"},
        {"Metric": "Training Steps", "Nerfacto (NeRF)": 5000, "Splatfacto v1 (5k)": 5000, "Splatfacto v2 (FINAL 15k)": 15000, "Final Gain": "step-000014999.ckpt"},
        {"Metric": "Prior Geometry", "Nerfacto (NeRF)": "None (Random)", "Splatfacto v1 (5k)": "None (Random)", "Splatfacto v2 (FINAL 15k)": "53,300 vertices", "Final Gain": "init_points.ply (from OBJ)"},
        {"Metric": "Exported Model", "Nerfacto (NeRF)": "point_cloud_vase_500k.ply", "Splatfacto v1 (5k)": "148,293 Gaussians", "Splatfacto v2 (FINAL 15k)": "ceramic_splatfacto_vase.ply", "Final Gain": "Default files set"}
    ])
    st.dataframe(df, hide_index=True, use_container_width=True)

    b1, b2 = st.columns(2)
    with b1:
        fig_psnr = go.Figure()
        fig_psnr.add_trace(go.Bar(
            x=["Nerfacto", "Splatfacto v1 (5k)", "Splatfacto v2 (FINAL 15k)"],
            y=[11.9713, 20.1740, 24.6751],
            marker_color=["#388bfd", "#f0883e", "#3fb950"],
            text=["11.97 dB", "20.17 dB", "24.68 dB"],
            textposition="auto"
        ))
        fig_psnr.update_layout(title="PSNR Quality (dB)", template="plotly_dark", paper_bgcolor="#0c1422", plot_bgcolor="#0c1422")
        st.plotly_chart(fig_psnr, use_container_width=True)

    with b2:
        fig_fps = go.Figure()
        fig_fps.add_trace(go.Bar(
            x=["Nerfacto", "Splatfacto v1 (5k)", "Splatfacto v2 (FINAL 15k)"],
            y=[0.105, 17.35, 22.88],
            marker_color=["#388bfd", "#f0883e", "#3fb950"],
            text=["0.10 FPS", "17.35 FPS", "22.88 FPS"],
            textposition="auto"
        ))
        fig_fps.update_layout(title="Rendering Performance (FPS) [217.9x Speedup!]", template="plotly_dark", paper_bgcolor="#0c1422", plot_bgcolor="#0c1422")
        st.plotly_chart(fig_fps, use_container_width=True)

# TAB 3: RETRAINING METHODOLOGY
with tab_retrain:
    st.subheader("Why Retraining Splatfacto v2 Reached 24.68 dB PSNR")
    st.markdown("""
    1. **Mesh Prior Initialization (`init_points.ply`):**
       - 53,300 surface vertices were extracted from `ceramic_vases_1.obj` and aligned to the camera frames.
       - Solved the initial geometric ambiguity of untextured ceramic surfaces.
    2. **Tripled Training Budget (15,000 Iterations):**
       - Scaled from 5,000 to **15,000 steps** (`step-000014999.ckpt`), allowing finer Gaussian densification.
    3. **Dynamic Densification & Pruning:**
       - **Step 1,000:** 102 duplicated, 1,672 split, 1,679 pruned ➔ 18,392 Gaussians.
       - **Step 2,000:** 548 duplicated, 28,872 split, 608 pruned ➔ 147,685 Gaussians.
       - **Final Count:** 218,537 Gaussians modeling directional specular highlights.
    4. **Web Viewer PLY Optimization:**
       - Dropped faint Gaussians (opacity $< 0.3$) and pruned divergent scales.
       - Shrunk the model from **14.8 MB down to 1.82 MB** for 60 FPS client rendering.
    """)

# TAB 4: HELD-OUT RENDERS
with tab_renders:
    st.subheader("Novel View Synthesis on 18 Held-Out Test Views")
    selected_idx = st.slider("Select Held-Out View Index (0 - 17):", 0, 17, 0)
    nerf_img = next(Path("/kaggle").rglob(f"eval_renders/*{selected_idx:04d}*.png"), None) if Path("/kaggle").exists() else None
    splat_img = next(Path("/kaggle").rglob(f"splatfacto*/*{selected_idx:04d}*.png"), None) if Path("/kaggle").exists() else None

    col1, col2 = st.columns(2)
    with col1:
        st.markdown("**Nerfacto (NeRF Baseline)**")
        if nerf_img: st.image(str(nerf_img))
        else: st.info(f"eval_img_{selected_idx:04d}.png")
    with col2:
        st.markdown("**Splatfacto v2 (Final Retrained 3DGS)**")
        if splat_img: st.image(str(splat_img))
        else: st.info(f"splatfacto_img_{selected_idx:04d}.png")

Overwriting app.py


In [28]:
# Cell 4: Launch Streamlit & Start Free Cloudflare Tunnel
import subprocess, time, re

# Kill old instances if any
!pkill -f streamlit || true

# Start Streamlit in background
subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true", shell=True)
time.sleep(3)

# Start Cloudflare Tunnel to expose the app publicly
proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stderr=subprocess.PIPE, text=True)

# Read the generated URL
url = None
for _ in range(25):
    line = proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break
    time.sleep(0.5)

print("\n" + "="*60)
if url:
    print("🎉 YOUR STREAMLIT APP IS RUNNING LIVE!")
    print(f"👉 CLICK THIS URL TO OPEN: {url}")
else:
    print("Could not auto-detect tunnel URL. Check if Internet is turned ON.")
print("="*60 + "\n")

  Stopping...




2026-11-15 16:40:38.442 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://35.192.208.131:8501


🎉 YOUR STREAMLIT APP IS RUNNING LIVE!
👉 CLICK THIS URL TO OPEN: https://ask-deutsch-replacement-adam.trycloudflare.com



2026-11-15 16:40:50.332 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-11-15 16:40:50.335 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-11-15 16:40:50.501 Serialization of dataframe to Arrow table was unsuccessful. Applying automatic fixes for column types to make the dataframe Arrow-compatible.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/streamlit/dataframe_util.py", line 967, in convert_pandas_df_to_arrow_table
    return pa.Table.from_pandas(df, preserve_index=preserve_index)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "pyarrow/table.pxi", line 4796, in pyarrow.lib.Table.from_pandas
  File "/usr/local/lib/python3.13/dist-packages/pyarrow/pandas_compat.py", line 651, 